# Confidential code sandbox — compute an average from Excel

This notebook calls a **confidential MCP code-interpreter** running inside an Azure
Container Instances **SEV-SNP TEE**. It authenticates as an **Entra service principal**
(the on-prem agent's identity), uploads an Excel spreadsheet, and asks the sandbox to
compute the average of each numeric column.

### Why this is confidential
- The spreadsheet is analyzed **inside a hardware-attested, memory-encrypted enclave**.
  The Azure host/operator and your infrastructure admins **cannot read the data in use**.
- Files live only in the enclave's **ephemeral in-memory workspace** — no external
  storage is mounted, so there is nothing for an admin to browse.

### Per-caller, per-session isolation
- Every call to `create_session` makes a **fresh workspace with its own Python venv**.
- The session is **owned by the calling identity** (the token's object id). No other
  caller — and no admin — can read another caller's session. Ownership is enforced
  *inside* the enclave.
- So each user gets a separate session, and each run gets a separate session **within**
  that user, too.

### Prerequisites
1. Network route to the private MCP endpoint (run this from on-prem **over the VPN**,
   or from inside the VNet). The endpoint is a private IP.
2. The on-prem agent's **service principal** granted the `Sandbox.Access` app role
   (already done for the app id below).
3. A `.env` file with your settings (copy `.env.example` to `.env`), including the
   client secret in `MCP_CLIENT_SECRET`.

## 1. Install dependencies

On Microsoft-managed devices public PyPI is blocked, so install via the package proxy.
Drop the `--index-url` if you are on an unrestricted network.

In [ ]:
%pip install --index-url https://packagefeedproxy.microsoft.io/pypi/simple \
    "azure-identity>=1.17" "mcp>=1.9,<2" "pandas>=2.0" "openpyxl>=3.1" "httpx>=0.27" "python-dotenv>=1.0"

## 2. Configuration

Configuration is loaded from a `.env` file (see `.env.example`). Copy it to `.env`
and fill in your values — never hardcode the tenant, client id, or secret.

In [ ]:
import os
from dotenv import load_dotenv

# Load values from a local .env file (does not override already-set env vars).
load_dotenv()


def _require(name):
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f"Missing required setting {name!r}. Set it in your .env file.")
    return value


# --- Entra identity -------------------------------------------------------
TENANT_ID = _require("MCP_TENANT_ID")

# The on-prem agent's service principal (client-credentials flow).
CLIENT_ID     = _require("MCP_CLIENT_ID")
CLIENT_SECRET = _require("MCP_CLIENT_SECRET")

# The confidential sandbox API app. With token v2 the access-token audience is
# this GUID; the request scope uses the api:// form.
API_APP_ID = _require("MCP_API_APP_ID")
SCOPE      = f"api://{API_APP_ID}/.default"

# --- MCP endpoint (private IP; reachable over the VPN / inside the VNet) ---
MCP_ENDPOINT = _require("MCP_ENDPOINT")

# --- Local spreadsheet to analyze -----------------------------------------
EXCEL_PATH = os.environ.get("MCP_EXCEL_PATH", "sample_sales.xlsx")

## 3. Make a sample spreadsheet (skip if you already have one)

Replace `EXCEL_PATH` above with your own file to analyze real data.

In [ ]:
import pandas as pd
from pathlib import Path

if not Path(EXCEL_PATH).exists():
    pd.DataFrame({
        "region":  ["East", "West", "North", "South"],
        "revenue": [120.0, 200.0, 150.0, 170.0],
        "units":   [12, 20, 15, 17],
    }).to_excel(EXCEL_PATH, index=False)

pd.read_excel(EXCEL_PATH)

## 4. Get an Entra token as the service principal

`azure-identity` performs the client-credentials flow. The resulting token has
`aud = <API_APP_ID>` and the `Sandbox.Access` role, which the enclave validates.

In [ ]:
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
access_token = credential.get_token(SCOPE).token
print("Acquired access token for", SCOPE)

## 5. Call the confidential sandbox

Flow: `initialize` -> `create_session` (new isolated workspace) -> `upload_file`
(the Excel bytes, base64) -> `run_python` (pandas computes the averages).

> **TLS note:** the enclave terminates TLS with a self-signed certificate, so the
> client skips CA/hostname verification. Authenticity is proven by `get_attestation`,
> not by the TLS chain. For production, issue a CA-signed cert for a DNS name that
> resolves to the private IP and enable verification.

In [ ]:
import base64, textwrap
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def _insecure_client(headers=None, timeout=None, auth=None):
    """httpx client that trusts the enclave's self-signed cert."""
    return httpx.AsyncClient(headers=headers, timeout=timeout, auth=auth, verify=False)


def _text(result):
    """Extract the text payload from a CallToolResult."""
    return result.content[0].text


async def compute_average(xlsx_path):
    headers = {"Authorization": f"Bearer {access_token}"}
    async with streamablehttp_client(
        MCP_ENDPOINT, headers=headers, httpx_client_factory=_insecure_client
    ) as (read, write, _):
        async with ClientSession(read, write) as mcp:
            await mcp.initialize()

            # A brand-new isolated session, owned by THIS service principal.
            session_id = _text(await mcp.call_tool("create_session", {}))
            print("session:", session_id)

            # Upload the spreadsheet as base64 bytes.
            data_b64 = base64.b64encode(Path(xlsx_path).read_bytes()).decode()
            await mcp.call_tool("upload_file", {
                "session_id": session_id, "path": "data.xlsx", "content_base64": data_b64,
            })

            # Ask the enclave to compute the average of every numeric column.
            code = textwrap.dedent('''
                import pandas as pd
                df = pd.read_excel("data.xlsx")
                means = df.select_dtypes("number").mean()
                print("Average per numeric column:")
                print(means.to_string())
            ''')
            result = _text(await mcp.call_tool("run_python", {
                "session_id": session_id, "code": code,
            }))
            print(result)
            return session_id


session_id = await compute_average(EXCEL_PATH)

## 6. (Optional) Prove it ran in a genuine enclave

`get_attestation` returns a signed SEV-SNP token from Microsoft Azure Attestation.
Verify it to confirm the result came from the real, untampered TEE.

In [ ]:
async def get_attestation():
    headers = {"Authorization": f"Bearer {access_token}"}
    async with streamablehttp_client(
        MCP_ENDPOINT, headers=headers, httpx_client_factory=_insecure_client
    ) as (read, write, _):
        async with ClientSession(read, write) as mcp:
            await mcp.initialize()
            return _text(await mcp.call_tool("get_attestation", {}))


print((await get_attestation())[:400], "...")

## 7. (Optional) Clean up this session

Sessions are ephemeral and vanish if the enclave restarts, but you can delete yours
explicitly. Only the owner (this service principal) is allowed to.

In [ ]:
async def delete_session(session_id):
    headers = {"Authorization": f"Bearer {access_token}"}
    async with streamablehttp_client(
        MCP_ENDPOINT, headers=headers, httpx_client_factory=_insecure_client
    ) as (read, write, _):
        async with ClientSession(read, write) as mcp:
            await mcp.initialize()
            return _text(await mcp.call_tool("delete_session", {"session_id": session_id}))


print(await delete_session(session_id))